In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1561_Mundka_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,200.49,299.68,3.85,26.86,17.42,30.78,10.14,1.06,19.54,...,NaN,11.19,72.28,1.09,89.13,0.00,0.00,51.79,982.20,NaN
1,2024-01-02,196.62,274.40,5.80,28.98,20.14,26.52,30.67,1.09,25.56,...,NaN,10.78,71.24,1.03,83.92,0.00,0.00,67.81,981.57,NaN
2,2024-01-03,235.95,352.47,13.27,38.55,31.32,32.79,12.30,1.34,23.13,...,NaN,10.67,76.81,0.79,140.97,0.00,0.00,53.89,981.19,NaN
3,2024-01-04,223.99,326.00,7.64,29.51,21.92,40.81,9.41,1.20,13.32,...,NaN,10.11,80.69,1.34,81.53,0.00,0.00,26.84,982.04,NaN
4,2024-01-05,162.22,244.70,3.29,30.00,18.62,43.48,11.45,1.13,12.36,...,NaN,11.22,80.73,1.00,106.23,0.00,0.00,27.72,981.25,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,177.79,286.50,11.87,48.33,34.91,43.44,6.85,1.15,19.47,...,NaN,14.87,88.29,0.73,60.88,1.01,1.01,10.92,979.50,NaN
362,2024-12-28,113.79,182.29,11.60,39.34,30.35,32.75,9.79,0.95,12.40,...,NaN,14.95,91.40,0.71,53.59,0.18,0.18,17.94,979.55,NaN
363,2024-12-29,103.12,165.29,2.37,22.38,13.82,24.55,8.57,0.81,26.44,...,NaN,14.00,88.98,0.72,65.29,0.00,0.00,39.64,979.61,NaN
364,2024-12-30,115.04,181.58,3.10,25.91,16.16,29.24,11.01,0.89,26.35,...,NaN,12.98,84.60,0.70,78.03,0.00,0.00,45.73,978.47,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         200.49        299.68        3.85        26.86   
1  2024-01-02         196.62        274.40        5.80        28.98   
2  2024-01-03         235.95        352.47       13.27        38.55   
3  2024-01-04         223.99        326.00        7.64        29.51   
4  2024-01-05         162.22        244.70        3.29        30.00   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      17.42        30.78       10.140        1.06          19.54   
1      20.14        26.52       11.535        1.09          25.56   
2      31.32        32.79       12.300        1.34          23.13   
3      21.92        40.81        9.410        1.20          13.32   
4      18.62        43.48       11.450        1.13          12.36   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             1.42             4.17    11.19   72.28      1

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.071800,0.159320,-1.084304,-1.307718,-1.276282,-0.293881,-0.278426,-0.127330,-0.721104,0.465186,-0.880245,-1.750624,0.793570,-0.487380,-0.964205,0.0,0.0,-1.357256,0.492268
1,2024-01-02,1.024084,-0.036299,-0.917148,-1.173205,-1.108904,-0.621355,0.149651,-0.004476,-0.486964,0.597982,-0.947205,-1.798895,0.721612,-0.685617,-1.048931,0.0,0.0,-0.890356,0.321444
2,2024-01-03,1.509011,0.567815,-0.276815,-0.565989,-0.420927,-0.139369,0.384403,1.019312,-0.581476,1.368199,0.421115,-1.811846,1.107000,-1.478569,-0.121181,0.0,0.0,-1.296052,0.218408
3,2024-01-04,1.361548,0.362987,-0.759422,-1.139576,-0.999369,0.477143,-0.502437,0.445991,-0.963023,0.704219,-0.530886,-1.877777,1.375458,0.338611,-1.087797,0.0,0.0,-2.084420,0.448884
4,2024-01-05,0.599942,-0.266121,-1.132307,-1.108486,-1.202439,0.682391,0.123568,0.159330,-1.000360,0.358949,-0.792905,-1.747092,1.378225,-0.784736,-0.686124,0.0,0.0,-2.058772,0.234677
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,0.791916,0.057332,-0.396824,0.054551,-0.200012,0.679316,-1.288013,0.241233,-0.723827,2.085298,0.089225,-1.317361,1.901302,-1.676806,-1.423608,0.0,0.0,-2.548405,-0.239833
362,2024-12-28,0.002815,-0.749057,-0.419968,-0.515863,-0.480617,-0.142444,-0.385829,-0.577796,-0.998805,0.358949,0.089225,-1.307942,2.116483,-1.742886,-1.542158,0.0,0.0,-2.343809,-0.226275
363,2024-12-29,-0.128743,-0.880605,-1.211170,-1.591974,-1.497813,-0.772793,-0.760204,-1.151117,-0.452738,-0.344871,0.118338,-1.419790,1.949043,-1.709846,-1.351892,0.0,0.0,-1.711366,-0.210007
364,2024-12-30,0.018227,-0.754551,-1.148594,-1.367996,-1.353818,-0.412264,-0.011453,-0.823505,-0.456238,0.119916,0.913129,-1.539879,1.645991,-1.775925,-1.144714,0.0,0.0,-1.533874,-0.519116


In [10]:
df.to_excel('Mundka2024.xlsx', index=False)